# Tmap 보행자 경유지 개수 상한 실측 (260731)

## 목적

Ploggo 경로 추천은 안전 핫스팟을 Tmap 보행자 경로안내 API의 경유지(`passList`)로 전달한다.
현재 K=3(핫스팟 3곳)으로 고정되어 있으나, UX 옵션으로 "짧게(K=3) / 보통(K=5) / 길게(K=7)"를
제공하는 방안이 검토 중이다. 이를 위해 **경유지를 몇 개까지 넣을 수 있는지** 확인이 필요하다.

## 배경

SK open API 공식 문서(https://openapi.sk.com)의 보행자 경로안내 항목에는 경유지 개수 상한이
명시되어 있지 않다. 일부 개발 사례에서 보행자 경로의 경유지가 최대 5개이며 초과 시 응답이
오지 않는다는 보고가 있으나, 공식 명세가 아니므로 직접 실측하여 확인한다.

만약 상한이 5개라면 K=7 옵션은 단일 호출로 구현할 수 없으며, 경로를 분할 호출하여
이어붙이는 방식이 필요하다. 이는 설계와 비용에 영향을 주므로 사전 확인이 중요하다.

## 방법

인천 도심에서 약 2km 구간의 출발지·도착지를 고정하고, 그 사이에 경유지를 0개부터 10개까지
균등 배치하여 순차 호출한다. 총 거리는 개수와 무관하게 유지되므로, 실패 발생 시 원인을
경유지 개수로 한정할 수 있다.

실패가 나와도 중단하지 않고 10개까지 진행한다. 특정 개수 이후 일관되게 실패하면 상한으로
판정하고, 실패가 불연속이면 일시적 오류로 보아 재실행을 권고한다.

## 실행 조건

- 총 API 호출 12회 (연결 확인 1회 + 0~10개 시험 11회)
- 호출 간격 0.6초, 총 소요 약 10~20초
- `.env`에 `TMAP_APP_KEY`가 설정되어 있어야 한다
- 결과는 `output_tmap_test/tmap_waypoint_limit_260731.csv`에 저장된다

## 판정 기준

| 결과 | 해석 | 후속 조치 |
| --- | --- | --- |
| 0~10개 전부 성공 | 상한이 10개 초과이거나 제한 없음 | K=3/5/7 모두 단일 호출로 구현 가능 |
| N개까지 성공, N+1부터 실패 | 상한 N개 | N 이하 옵션만 제공하거나 분할 호출 설계 |
| 실패가 불연속 | 일시 오류 또는 좌표 문제 | 동일 조건 재실행하여 재현성 확인 |


In [1]:
# ==========================================================================
# 셀 1: 환경 준비 및 인증 키 로드
# ==========================================================================
# Tmap 보행자 경로안내 API의 경유지(passList) 개수 상한을 실측한다.
# 공식 문서에 상한이 명시되어 있지 않으며, 실사용 보고 중 5개 제한 언급이 있으나
# 확인이 필요하다. 경유지를 0개부터 10개까지 늘려가며 응답 성공 여부를 기록한다.
#
# 참조: 보행자 경로안내 https://apis.openapi.sk.com/tmap/routes/pedestrian
#       SK open API 포털 https://openapi.sk.com

import os
import time
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
TMAP_APP_KEY = os.getenv("TMAP_APP_KEY")
if not TMAP_APP_KEY:
    raise RuntimeError(".env에서 TMAP_APP_KEY를 찾을 수 없다. 파일 위치와 키 이름을 확인한다.")

print(f"[인증 키] 로드 완료 (길이 {len(TMAP_APP_KEY)})")
print(f"[작업 디렉토리] {os.getcwd()}")

TMAP_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
REQUEST_INTERVAL_SEC = 0.6   # 연속 호출 간 대기 (API 부하 완화)
MAX_WAYPOINTS = 10           # 시험할 최대 경유지 수



[인증 키] 로드 완료 (길이 40)
[작업 디렉토리] c:\Users\User\Documents\GitHub\ploggo-algorithm-test


In [2]:
# ==========================================================================
# 셀 2: 시험용 좌표 생성 (인천 도심, 도보 가능 범위)
# ==========================================================================
# 실제 서비스 조건과 유사하도록 인천 도심에서 O -> D 약 2km 구간을 잡고,
# 그 사이에 경유지를 균등 배치한다. 경유지가 늘어도 총 거리는 유지되므로
# 실패 원인을 '개수'로 한정할 수 있다.

ORIGIN      = (37.4563, 126.7052)   # (lat, lon) 인천시청 인근
DESTINATION = (37.4700, 126.6900)   # 약 2km 북서

def make_waypoints(n):
    """O와 D 사이 직선을 n+1등분하여 경유지 n개를 생성한다."""
    if n == 0:
        return []
    pts = []
    for i in range(1, n + 1):
        t = i / (n + 1)
        lat = ORIGIN[0] + (DESTINATION[0] - ORIGIN[0]) * t
        lon = ORIGIN[1] + (DESTINATION[1] - ORIGIN[1]) * t
        # 직선상 완전 일치를 피하기 위해 미세 오프셋 (도로 스냅 유도)
        offset = 0.0008 * (1 if i % 2 == 0 else -1)
        pts.append((lat + offset, lon + offset))
    return pts

print(f"[출발지] {ORIGIN}")
print(f"[도착지] {DESTINATION}")
for n in [0, 3, 5, 10]:
    print(f"  경유지 {n:>2}개 예시: {[(round(a,5), round(b,5)) for a, b in make_waypoints(n)][:3]}"
          f"{' ...' if n > 3 else ''}")



[출발지] (37.4563, 126.7052)
[도착지] (37.47, 126.69)
  경유지  0개 예시: []
  경유지  3개 예시: [(37.45893, 126.7006), (37.46395, 126.6984), (37.46578, 126.693)]
  경유지  5개 예시: [(37.45778, 126.70187), (37.46167, 126.70093), (37.46235, 126.6968)] ...
  경유지 10개 예시: [(37.45675, 126.70302), (37.45959, 126.70324), (37.45924, 126.70025)] ...


In [3]:
# ==========================================================================
# 셀 3: 단일 호출 함수 정의
# ==========================================================================
# passList 형식: "경도,위도_경도,위도_..." (경도가 앞, 언더스코어로 구분)
# 좌표 순서를 위경도로 잘못 넣으면 엉뚱한 결과가 나오므로 주의한다.

def call_pedestrian_route(origin, destination, waypoints=None, timeout=15):
    """보행자 경로안내 API를 1회 호출하고 결과를 요약해 반환한다."""
    body = {
        "startX": origin[1], "startY": origin[0],
        "endX": destination[1], "endY": destination[0],
        "startName": "출발", "endName": "도착",
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
        "searchOption": "0", "sort": "index",
    }
    if waypoints:
        body["passList"] = "_".join(f"{lon},{lat}" for lat, lon in waypoints)

    headers = {"appKey": TMAP_APP_KEY,
               "Content-Type": "application/json",
               "Accept": "application/json"}

    t0 = time.time()
    try:
        resp = requests.post(TMAP_URL, headers=headers, json=body, timeout=timeout)
    except requests.RequestException as e:
        return {"ok": False, "http_status": None, "reason": f"요청 예외: {type(e).__name__}",
                "elapsed_ms": (time.time() - t0) * 1000, "distance_m": None, "n_points": 0}

    elapsed_ms = (time.time() - t0) * 1000

    if resp.status_code != 200:
        # 오류 응답 본문에 원인이 담기는 경우가 있으므로 앞부분을 보존한다
        detail = resp.text[:300].replace("\n", " ")
        return {"ok": False, "http_status": resp.status_code, "reason": detail,
                "elapsed_ms": elapsed_ms, "distance_m": None, "n_points": 0}

    try:
        data = resp.json()
    except json.JSONDecodeError:
        return {"ok": False, "http_status": 200, "reason": "JSON 파싱 실패",
                "elapsed_ms": elapsed_ms, "distance_m": None, "n_points": 0}

    features = data.get("features", [])
    if not features:
        return {"ok": False, "http_status": 200, "reason": "features 비어 있음",
                "elapsed_ms": elapsed_ms, "distance_m": None, "n_points": 0}

    # 총 거리는 첫 Point feature의 properties.totalDistance에 담긴다
    total_distance = None
    n_points = 0
    for f in features:
        props = f.get("properties", {})
        if total_distance is None and "totalDistance" in props:
            total_distance = props["totalDistance"]
        geom = f.get("geometry", {})
        if geom.get("type") == "LineString":
            n_points += len(geom.get("coordinates", []))

    return {"ok": True, "http_status": 200, "reason": "OK",
            "elapsed_ms": elapsed_ms, "distance_m": total_distance, "n_points": n_points}


# 연결 확인 (경유지 없이 1회)
print("[연결 확인] 경유지 0개로 시험 호출")
_probe = call_pedestrian_route(ORIGIN, DESTINATION)
print(f"  결과: {'성공' if _probe['ok'] else '실패'} | HTTP {_probe['http_status']} "
      f"| 거리 {_probe['distance_m']}m | {_probe['elapsed_ms']:.0f}ms")
if not _probe["ok"]:
    print(f"  사유: {_probe['reason']}")
    print("  키 또는 네트워크 문제일 수 있다. 아래 셀 진행 전 확인이 필요하다.")



[연결 확인] 경유지 0개로 시험 호출
  결과: 성공 | HTTP 200 | 거리 2987m | 401ms


In [4]:
# ==========================================================================
# 셀 4: 경유지 0~10개 순차 시험
# ==========================================================================
# 각 개수마다 1회씩 호출한다. 실패가 나와도 중단하지 않고 끝까지 진행하여,
# 특정 개수에서만 실패하는지 그 이후 전부 실패하는지를 구분한다.

import pandas as pd

print("=" * 78)
print(f"[Tmap 보행자 경유지 개수 시험 — 0 ~ {MAX_WAYPOINTS}개]")
print("=" * 78)

results = []
for n in range(0, MAX_WAYPOINTS + 1):
    wps = make_waypoints(n)
    r = call_pedestrian_route(ORIGIN, DESTINATION, waypoints=wps)
    results.append({
        "경유지수": n,
        "성공": r["ok"],
        "HTTP": r["http_status"],
        "거리_m": r["distance_m"],
        "경로점수": r["n_points"],
        "응답_ms": round(r["elapsed_ms"]),
        "사유": r["reason"][:60] if not r["ok"] else "",
    })
    status = "성공" if r["ok"] else "실패"
    extra = f"거리 {r['distance_m']}m" if r["ok"] else f"HTTP {r['http_status']} | {r['reason'][:50]}"
    print(f"  경유지 {n:>2}개: {status} | {extra}")
    time.sleep(REQUEST_INTERVAL_SEC)

df = pd.DataFrame(results)



[Tmap 보행자 경유지 개수 시험 — 0 ~ 10개]
  경유지  0개: 성공 | 거리 2987m
  경유지  1개: 성공 | 거리 3034m
  경유지  2개: 성공 | 거리 3484m
  경유지  3개: 성공 | 거리 3181m
  경유지  4개: 성공 | 거리 3250m
  경유지  5개: 성공 | 거리 3816m
  경유지  6개: 실패 | HTTP 400 | { "error":{  "id":"400",  "category":"tmap",  "cod
  경유지  7개: 실패 | HTTP 400 | { "error":{  "id":"400",  "category":"tmap",  "cod
  경유지  8개: 실패 | HTTP 400 | { "error":{  "id":"400",  "category":"tmap",  "cod
  경유지  9개: 실패 | HTTP 400 | { "error":{  "id":"400",  "category":"tmap",  "cod
  경유지 10개: 실패 | HTTP 400 |        �VPJ-*�/R��VPP�LQ�R210P�r�KR��*�B%��


In [5]:
# ==========================================================================
# 셀 5: 결과 정리 및 상한 판정
# ==========================================================================

print("\n" + "=" * 78)
print("[시험 결과]")
print("=" * 78)
print(df.to_string(index=False))

ok_counts = df[df["성공"]]["경유지수"].tolist()
fail_counts = df[~df["성공"]]["경유지수"].tolist()

print("\n" + "-" * 78)
print("[판정]")
if not ok_counts:
    print("  전 구간 실패. 인증 키·네트워크·좌표를 먼저 점검한다.")
elif not fail_counts:
    print(f"  0~{MAX_WAYPOINTS}개 전부 성공. 상한은 {MAX_WAYPOINTS}개보다 크거나 제한이 없다.")
    print(f"  서비스에 필요한 범위(K=3~7)는 모두 사용 가능하다.")
else:
    max_ok = max(ok_counts)
    min_fail = min(fail_counts)
    print(f"  성공: {ok_counts}")
    print(f"  실패: {fail_counts}")
    if max_ok + 1 == min_fail and all(c > max_ok for c in fail_counts):
        print(f"  => 경유지 상한은 {max_ok}개로 판단된다 ({min_fail}개부터 일관되게 실패).")
    else:
        print(f"  => 실패가 연속적이지 않다. 개수 제한이 아닌 다른 원인(일시 오류, 좌표 문제)일 수 있다.")
        print(f"     동일 조건으로 재실행하여 재현성을 확인한다.")

    # K 옵션 가능 여부 판정
    print("\n  [UX 옵션 적용 가능 범위]")
    for k, label in [(3, "짧게"), (5, "보통"), (7, "길게")]:
        avail = k in ok_counts
        print(f"    K={k} ({label}): {'사용 가능' if avail else '사용 불가'}")

# 결과 저장
out_dir = Path("./output_tmap_test")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "tmap_waypoint_limit_260731.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n[CSV 저장] {csv_path.resolve()}")
print(f"[총 API 호출] {MAX_WAYPOINTS + 2}회 (연결 확인 1회 포함)")




[시험 결과]
 경유지수    성공  HTTP   거리_m  경로점수  응답_ms                                                           사유
    0  True   200 2987.0   157    221                                                             
    1  True   200 3034.0   181    344                                                             
    2  True   200 3484.0   203    215                                                             
    3  True   200 3181.0   168    227                                                             
    4  True   200 3250.0   200    231                                                             
    5  True   200 3816.0   204    253                                                             
    6 False   400    NaN     0    203 { "error":{  "id":"400",  "category":"tmap",  "code":"1100",
    7 False   400    NaN     0    141 { "error":{  "id":"400",  "category":"tmap",  "code":"1100",
    8 False   400    NaN     0    159 { "error":{  "id":"400",  "category":"tmap",  "code":"1100",
 